# Test training

In [1]:
import os
import gc
import json
import ast
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
from datasets import Dataset, DatasetDict, Sequence, Value
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from scipy.special import expit

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.backends.cuda.matmul.allow_tf32 = True
device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

MAX_LEN = 512
THRESH = 0.5

2026-04-13 04:23:24.117065: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-13 04:23:24.415173: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-13 04:23:25.618588: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2026-04-13 04:23:25.618718: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] 

In [2]:
def to_list_labels(x):
    """Parsea level1_codes desde string de lista o valor suelto."""
    if isinstance(x, (list, np.ndarray)):
        return [str(i) for i in x]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, (list, tuple, set)):
                return [str(i) for i in parsed]
        except (ValueError, SyntaxError):
            pass
        return [x.strip()]
    return [str(x)]


def load_and_prepare_data(data_dir="data/pubmed_mesh"):
    """Carga los CSV, crea input_text y parsea labels."""
    df_train = pd.read_csv(os.path.join(data_dir, "train_multilabel.csv"))
    df_dev   = pd.read_csv(os.path.join(data_dir, "val_multilabel.csv"))
    df_test  = pd.read_csv(os.path.join(data_dir, "test_multilabel.csv"))

    for df in (df_train, df_dev, df_test):
        # Crear input_text = title + abstract
        df["input_text"] = (
            df["title"].fillna("") + " " + df["spanish_abstract"].fillna("")
        ).str.strip()
        # Parsear labels
        df["labels"] = df["level1_codes"].apply(to_list_labels)

    # Quitar filas sin texto o sin etiquetas
    clean = []
    for name, df in (("train", df_train), ("dev", df_dev), ("test", df_test)):
        n0 = len(df)
        df = df.dropna(subset=["input_text"])
        df = df[df["input_text"].str.len() > 0]
        df = df[df["labels"].map(len) > 0]
        print(f"{name}: {n0} → {len(df)} (filas válidas)")
        clean.append(df)

    df_train, df_dev, df_test = clean

    # MultiLabelBinarizer (fit solo en train)
    mlb = MultiLabelBinarizer()
    Y_train = mlb.fit_transform(df_train["labels"])
    Y_dev   = mlb.transform(df_dev["labels"])
    Y_test  = mlb.transform(df_test["labels"])

    id2label = {i: lab for i, lab in enumerate(mlb.classes_)}
    label2id = {lab: i for i, lab in id2label.items()}
    num_labels = len(mlb.classes_)
    print(f"Número de etiquetas: {num_labels} → {list(mlb.classes_)}")

    # Columna target como float32
    df_train = df_train.assign(target=[row.astype("float32").tolist() for row in Y_train])
    df_dev   = df_dev.assign(target=[row.astype("float32").tolist() for row in Y_dev])
    df_test  = df_test.assign(target=[row.astype("float32").tolist() for row in Y_test])

    return df_train, df_dev, df_test, id2label, label2id, num_labels


def build_datasets(df_train, df_dev, df_test, tokenizer):
    """Tokeniza y crea DatasetDict listo para Trainer."""
    cols_keep = ["input_text", "target"]
    ds = DatasetDict({
        "train":      Dataset.from_pandas(df_train[cols_keep], preserve_index=False),
        "validation": Dataset.from_pandas(df_dev[cols_keep],   preserve_index=False),
        "test":       Dataset.from_pandas(df_test[cols_keep],  preserve_index=False),
    })

    def tok_fn(batch):
        return tokenizer(batch["input_text"], truncation=True, max_length=MAX_LEN)

    ds_tok = ds.map(tok_fn, batched=True, remove_columns=["input_text"])

    for split in ds_tok:
        ds_tok[split] = ds_tok[split].add_column("labels", ds[split]["target"])
        ds_tok[split] = ds_tok[split].remove_columns(["target"])

    ds_tok = ds_tok.cast_column("labels", Sequence(Value("float32")))
    return ds_tok


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    labels = (np.asarray(labels) > 0.5).astype(int)
    probs  = expit(logits)
    preds  = (probs >= THRESH).astype(int)

    f1_micro    = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro    = f1_score(labels, preds, average="macro", zero_division=0)
    exact_match = (preds == labels).all(axis=1).mean().item()

    return {"f1_micro": f1_micro, "f1_macro": f1_macro, "exact_match": exact_match}

In [3]:
batchs = [16, 32]
lrs = [1e-5, 2e-5, 3e-5]
epochs = 10
weight_decay = 0.1
warmup = 0.06
lr_decay = "linear"

In [5]:
def train_beto(batchs, lrs, epochs, weight_decay, warmup, lr_decay):
    all_results = []

    df_train, df_dev, df_test, id2label, label2id, num_labels = load_and_prepare_data()

    tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
    ds_tok = build_datasets(df_train, df_dev, df_test, tokenizer)
    data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt", pad_to_multiple_of=8)

    for batch in batchs:
        for lr in lrs:
            print(f"\n{'='*60}")
            print(f"BETO — lr={lr}, batch={batch}")
            print(f"{'='*60}")

            out_dir = f"models/pubmed_beto_bs{batch}_lr{lr}".replace(".", "p")

            config = AutoConfig.from_pretrained(
                "dccuchile/bert-base-spanish-wwm-cased",
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id,
                problem_type="multi_label_classification",
            )
            model = AutoModelForSequenceClassification.from_pretrained(
                "dccuchile/bert-base-spanish-wwm-cased", config=config
            )
            model.gradient_checkpointing_enable()
            model.config.use_cache = False
            model.to(device)

            args = TrainingArguments(
                output_dir=out_dir,
                evaluation_strategy="epoch",
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model="f1_micro",
                greater_is_better=True,
                learning_rate=lr,
                per_device_train_batch_size=batch,
                per_device_eval_batch_size=batch,
                num_train_epochs=epochs,
                weight_decay=weight_decay,
                logging_dir="./logs",
                logging_steps=50,
                save_total_limit=1,
                seed=444,
                fp16=True,
                warmup_ratio=warmup,
                lr_scheduler_type=lr_decay,
                report_to=["none"],
            )

            trainer = Trainer(
                model=model,
                args=args,
                train_dataset=ds_tok["train"],
                eval_dataset=ds_tok["validation"],
                tokenizer=tokenizer,
                data_collator=data_collator,
                compute_metrics=compute_metrics,
            )

            trainer.train()

            # Guardar mejor modelo
            best_dir = os.path.join(out_dir, "best")
            os.makedirs(best_dir, exist_ok=True)
            trainer.save_model(best_dir)
            tokenizer.save_pretrained(best_dir)

            # Evaluar
            dev_metrics  = trainer.evaluate(eval_dataset=ds_tok["validation"])
            test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])

            result = {
                "model_name": "beto",
                "batch_size": batch,
                "learning_rate": lr,
                "epochs": epochs,
                "weight_decay": weight_decay,
                "warmup_ratio": warmup,
                "lr_scheduler": lr_decay,
                "best_checkpoint": trainer.state.best_model_checkpoint,
                "val_exact_match": float(dev_metrics.get("eval_exact_match", np.nan)),
                "val_f1_micro":    float(dev_metrics.get("eval_f1_micro", np.nan)),
                "val_f1_macro":    float(dev_metrics.get("eval_f1_macro", np.nan)),
                "test_exact_match": float(test_metrics.get("eval_exact_match", np.nan)),
                "test_f1_micro":    float(test_metrics.get("eval_f1_micro", np.nan)),
                "test_f1_macro":    float(test_metrics.get("eval_f1_macro", np.nan)),
            }

            all_results[:] = sorted(
                all_results + [result],
                key=lambda r: r["test_f1_micro"],
                reverse=True,
            )

            json_path = "results/pubmed_beto.json"
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(all_results, f, ensure_ascii=False, indent=2)

            print(f"  VAL  F1-micro={result['val_f1_micro']:.4f}  F1-macro={result['val_f1_macro']:.4f}")
            print(f"  TEST F1-micro={result['test_f1_micro']:.4f}  F1-macro={result['test_f1_macro']:.4f}")

            del model, trainer
            gc.collect()
            torch.cuda.empty_cache()

    return all_results

In [ ]:
# 1) Entrenar BETO
print("\n" + "█" * 60)
print("  ENTRENANDO BETO")
print("█" * 60)
results_beto = train_beto(batchs, lrs, epochs, weight_decay, warmup, lr_decay)


████████████████████████████████████████████████████████████
  ENTRENANDO BETO
████████████████████████████████████████████████████████████
train: 18600 → 18600 (filas válidas)
dev: 4650 → 4650 (filas válidas)
test: 5813 → 5813 (filas válidas)
Número de etiquetas: 15 → ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'Z']


Map:   0%|          | 0/18600 [00:00<?, ? examples/s]

Map:   0%|          | 0/4650 [00:00<?, ? examples/s]

Map:   0%|          | 0/5813 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/18600 [00:00<?, ? examples/s]

...

Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchi

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Exact Match
1,0.240900,0.241544,0.555070,0.351998,0.255699
2,0.209600,0.213117,0.599802,0.431197,0.295699
3,0.188400,0.204838,0.634682,0.480649,0.318280
4,0.166300,0.204504,0.651893,0.512863,0.324946
5,0.137000,0.209310,0.656496,0.532278,0.326022
6,0.129900,0.214002,0.640203,0.518285,0.327957
7,0.114700,0.219687,0.643264,0.523735,0.330968
8,0.100100,0.225870,0.642866,0.528234,0.320645
9,0.097200,0.229055,0.646566,0.531547,0.322796
10,0.086400,0.229548,0.649098,0.535333,0.323226


  VAL  F1-micro=0.6565  F1-macro=0.5323
  TEST F1-micro=0.6650  F1-macro=0.5445

BETO — lr=2e-05, batch=16


Some weights of the model checkpoint at dccuchile/bert-base-spanish-wwm-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchi

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Exact Match
1,0.228000,0.228191,0.580274,0.393682,0.273548
2,0.196200,0.203679,0.628906,0.482975,0.316129
